# Fork Existing Trial

Start from the best logged MLflow run, materialize its model source into a local project draft, then run that draft through the same runner used by AutoML.

In [2]:
from __future__ import annotations

import os

import pandas as pd
from IPython.display import display

import automl
from automl import experiment, trial

DRY_RUN = True
NAMESPACE = os.getenv("AUTOML_NOTEBOOK_NAMESPACE", "")
BASE_FORK_SLUG = "notebook_fork"


/Users/zhengisamazing/1.python_dir/brigit/automl_dev-refactor/.venv/lib/python3.13/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [3]:
active = automl.use_project(dry_run=DRY_RUN, namespace=NAMESPACE)
config = active.config
display(
    {
        "project": active.project_name,
        "repo_root": str(config.repo_root),
        "project_dir": str(config.project_dir),
        "experiment": active.active_experiment_id,
        "dry_run": active.dry_run,
        "namespace": active.namespace or "<none>",
    }
)

leaderboard = experiment.leaderboard(training_origin="all", n=5, session=active)
rows = list(leaderboard.rows)
leaderboard_df = pd.DataFrame([row.to_dict() for row in rows])
leaderboard_df if not leaderboard_df.empty else pd.DataFrame(
    {"message": ["No successful trials to fork yet. Run 3.1_run_agent_automl.ipynb or 3.2_author_new_trial.ipynb first."]}
)


{'project': 'example_homecredit',
 'repo_root': '/Users/zhengisamazing/1.python_dir/brigit/automl_dev-refactor',
 'project_dir': '/Users/zhengisamazing/1.python_dir/brigit/automl_dev-refactor/projects/example_homecredit',
 'experiment': 'example-homecredit',
 'dry_run': True}

,schema_version,run_id,slug,strategy,status,primary_metric_name,primary_metric_value,started_at,ended_at,parent_run_id,dataset_hash,trial_number,hypothesis,training_origin,training_time_s,n_features
0,1,b29ea9099bc54a2582b84dfeb550513d,example_notebook_2_elasticnet,baseline,FINISHED,eval.test.auc,0.631579,2026-06-02T07:04:13.715000Z,2026-06-02T07:04:53.140000Z,None,sha256:8471c462af97cf485e43b918cf671bcf44bccde...,1,An elastic-net logistic regression applied to ...,automl,44.016953,None
1,1,b8b52c8012ea47638beee888781c37bf,notebook_baseline_2,human_baseline,RUNNING,eval.test.auc,0.500000,2026-06-02T07:07:56.956000Z,2026-06-02T07:08:00.700000Z,None,sha256:8471c462af97cf485e43b918cf671bcf44bccde...,2,Notebook-authored baseline.,human,NaN,None


In [4]:
seed = rows[0] if rows else None
seed.to_dict() if seed is not None else pd.DataFrame(
    {"message": ["No seed run is available yet."]}
)


{'schema_version': 1,
 'run_id': 'b29ea9099bc54a2582b84dfeb550513d',
 'slug': 'example_notebook_2_elasticnet',
 'strategy': 'baseline',
 'status': 'FINISHED',
 'primary_metric_name': 'eval.test.auc',
 'primary_metric_value': 0.631578947368421,
 'started_at': '2026-06-02T07:04:13.715000Z',
 'ended_at': '2026-06-02T07:04:53.140000Z',
 'parent_run_id': None,
 'dataset_hash': 'sha256:8471c462af97cf485e43b918cf671bcf44bccde57256d40d344e7ef4cafba91a',
 'trial_number': 1,
 'hypothesis': 'An elastic-net logistic regression applied to the WOE-encoded feature pool will establish a regularized linear baseline that is interpretable and well-suited to sparse credit risk features, providing a reliable AUC reference for the example_homecredit project.',
 'training_origin': 'automl',
 'training_time_s': 44.01695312501397,
 'n_features': None}

In [5]:
fork_dir = None
fork_slug = None

if seed is not None:
    slug_index = 1
    while fork_dir is None:
        candidate_slug = BASE_FORK_SLUG if slug_index == 1 else f"{BASE_FORK_SLUG}_{slug_index}"
        try:
            fork_dir = trial.fork(
                candidate_slug,
                seed="best",
                strategy="manual_fork",
                hypothesis="Manual notebook fork.",
                session=active,
            )
            fork_slug = candidate_slug
        except FileExistsError:
            slug_index += 1

{
    "fork_slug": fork_slug,
    "fork_dir": str(fork_dir) if fork_dir is not None else None,
} if fork_dir is not None else pd.DataFrame(
    {"message": ["Create a successful trial first, then rerun this cell to fork it."]}
)


{'fork_slug': 'notebook_fork',
 'fork_dir': '/Users/zhengisamazing/1.python_dir/brigit/automl_dev-refactor/projects/example_homecredit/experiments/dry_run/example_homecredit/example-homecredit/notebook_fork'}

In [6]:
if fork_dir is None:
    fork_metadata = pd.DataFrame({"message": ["No fork directory yet."]})
else:
    print(f"Edit this model: {fork_dir / 'model.py'}")
    print("Then run it with: uv run automl --project example_homecredit trial run <draft_dir>")
    fork_metadata = (fork_dir / "metadata.json").read_text()

fork_metadata


Edit this model: /Users/zhengisamazing/1.python_dir/brigit/automl_dev-refactor/projects/example_homecredit/experiments/dry_run/example_homecredit/example-homecredit/notebook_fork/model.py
Then run it with: uv run automl --project example_homecredit trial run <draft_dir>


'{\n  "schema_version": 1,\n  "slug": "notebook_fork",\n  "strategy": "manual_fork",\n  "hypothesis": "Manual notebook fork.",\n  "training_origin": "human",\n  "created_at": "2026-06-02T07:08:25.416124+00:00",\n  "project_name": "example_homecredit",\n  "project_package": "projects.example_homecredit",\n  "experiment_id": "example-homecredit",\n  "seed": {\n    "schema_version": 1,\n    "selector": "best",\n    "run_id": "b29ea9099bc54a2582b84dfeb550513d",\n    "trial_id": "example_notebook_2_elasticnet",\n    "metric_name": "eval.test.auc",\n    "metric_value": 0.631578947368421,\n    "strategy": "baseline",\n    "model_source": {\n      "source": "mlflow",\n      "artifact_path": "model/code"\n    }\n  }\n}'

In [7]:
# from automl.runner import run_trial
# result = run_trial(fork_dir, session=active)
# result.status, result.trial_id, result.trial_number, result.run_id
